# 21cmCU Quickstart Notebook (`v0.6.0`)

This notebook walks through four core workflows with native C/CUDA acceleration and zero CuPy dependency:
1. Hardware & CUDA diagnostics (`py21cmcu.doctor()`)
2. Single-GPU Coeval box + 1D power spectrum (`L-INTEGRAL + TS`, `CUDA_LAZY_D2H=True`)
3. GPU Lightcone + RSDs + Horizon Wedge + Thermal Noise + `DeviceLZF` HDF5 archival
4. Multi-GPU slab IC (`CUDA_DEVICES`) & resumable SBI batch sweeps (`run_batch`)

In [ ]:
import tempfile
import numpy as np
import py21cmcu as p21

rep = p21.doctor(as_json=False)
print('21cmCU version:', rep['versions']['py21cmcu'])
print('CUDA compiled / available / GPUs:', rep['cuda']['compiled'], rep['cuda']['available'], rep['cuda']['device_count'])
for gpu in rep['cuda']['gpus']:
    print(f"  GPU {gpu['index']}: {gpu['name']} ({gpu['memory_mib']} MiB, sm_{gpu['compute_cap']})")

In [ ]:
inputs = p21.InputParameters(random_seed=42, node_redshifts=[8.0, 9.5, 11.0, 12.5]).evolve_input_structs(
    DIM=128,
    HII_DIM=64,
    BOX_LEN=128.0,
    N_THREADS=4,
    SOURCE_MODEL='L-INTEGRAL',
    USE_TS_FLUCT=True,
    Z_HEAT_MAX=12.0,
    USE_UPPER_STELLAR_TURNOVER=False,
    PHOTON_CONS_TYPE='no-photoncons',
)

with tempfile.TemporaryDirectory() as tmpdir, p21.use_mode('fast', use_cuda=True, lazy_d2h=True, direc=tmpdir):
    coeval = p21.run_coeval(inputs=inputs, out_redshifts=[8.0], write=False, regenerate=True)[0]
    ps1 = p21.power_spectrum_1d(coeval.brightness_temp, box_length=128.0, n_bins=12, remove_mean=True)
    print(f"<x_HI> = {float(np.mean(coeval.neutral_fraction)):.4f}, <Tb> = {float(np.mean(coeval.brightness_temp)):.2f} mK")
    print('CUDA registry hits:', p21.cuda_stats()['registry_hits'])
    p21.cleanup_cuda()

In [ ]:
inputs_lc = p21.InputParameters(random_seed=101, node_redshifts=[6.5, 7.5, 8.5, 9.5, 11.0]).evolve_input_structs(
    DIM=128,
    HII_DIM=64,
    BOX_LEN=128.0,
    N_THREADS=4,
    SOURCE_MODEL='L-INTEGRAL',
    USE_TS_FLUCT=True,
    Z_HEAT_MAX=10.5,
    USE_UPPER_STELLAR_TURNOVER=False,
    PHOTON_CONS_TYPE='no-photoncons',
)
lc_def = p21.RectilinearLightconer.between_redshifts(
    min_redshift=7.0,
    max_redshift=10.0,
    resolution=inputs_lc.simulation_options.cell_size,
    quantities=('brightness_temp', 'neutral_fraction'),
    cosmo=inputs_lc.cosmo_params.cosmo,
)

with tempfile.TemporaryDirectory() as tmpdir, p21.use_mode('fast', use_cuda=True, lazy_d2h=True, direc=tmpdir):
    lc = p21.run_lightcone(lightconer=lc_def, inputs=inputs_lc, include_dvdr_in_tau21=True, apply_rsds=True, regenerate=True, write=False)
    tb = np.asarray(lc.lightcones['brightness_temp_with_rsds'], dtype=np.float32)[:, :, :64]
    obs = p21.apply_wedge_filter_3d(p21.add_thermal_noise_3d(tb, 128.0, sigma_mk=2.0, seed=7), 128.0, redshift=8.5, buffer_h_mpc=0.05)
    p21.save_lightcone(lc, 'quickstart_lc.h5', compression='device-lzf')
    print('Saved quickstart_lc.h5, filtered RMS [mK]:', float(np.std(obs)))
    p21.cleanup_cuda()

In [ ]:
# Sobol parameter sweep + resumable HDF5 batch pipeline (adjust samples/devices as needed)
bounds = {'F_STAR10_ACG': (-1.8, -0.7), 'F_ESC10_ACG': (-1.8, -0.5)}
rows = p21.generate_sobol_parameter_grid(bounds, n_samples=4, seed=42)
tasks = [
    p21.BatchTask(
        task_id=f'demo_{i:02d}',
        inputs=p21.InputParameters(random_seed=int(r['random_seed']), node_redshifts=[8.0]).evolve_input_structs(
            DIM=64, HII_DIM=32, BOX_LEN=64.0, N_THREADS=2, USE_TS_FLUCT=False, RECOMB_MODEL='none',
            F_STAR10_ACG=float(r['F_STAR10_ACG']), F_ESC10_ACG=float(r['F_ESC10_ACG']),
        ),
        mode='coeval',
        out_redshifts=(8.0,),
        cuda_mode='fast',
        wedge_filter=True,
        noise_sigma_mk=2.0,
    )
    for i, r in enumerate(rows)
]
results = p21.run_batch(tasks, devices=(0,), workers_per_gpu=1, output_h5='quickstart_sbi.h5', checkpoint_h5='quickstart_sbi.h5', resume=True, in_process=True)
print('Batch ok:', [r.ok for r in results])